# 03 · Gold — um dado, dois consumos

```
silver ─► gold.vendas_diarias ─┐
silver ─► gold.clientes_360    ├─► Iceberg ─► Postgres (schema gold) ─► Metabase   [BI]
silver ─► gold.avaliacoes_resumo┘

silver.avaliacoes ─► gold.avaliacao_chunks ─► Iceberg ─► pgvector (schema vetorial) [próximo módulo]
```

A Gold tem **dois públicos**: pessoas (painéis) e, em seguida, um modelo de linguagem (busca por significado).
Em ambos, o **Iceberg continua sendo a fonte da verdade**; o Postgres é uma *cópia para servir*.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))          # para importar a pasta common/
from common import lake_local as L, medalhao as M
from pyspark.sql import functions as F
spark = L.get_spark()                              # PySpark local: sem cluster
print("Spark", spark.version, "| master:", spark.sparkContext.master, "| storage:", L.STORAGE)

## Parte A · Gold para BI

In [ ]:
vd  = M.build_vendas_diarias(spark)
c360 = M.build_clientes_360(spark)
ares = M.build_avaliacoes_resumo(spark)
for t in ("vendas_diarias","clientes_360","avaliacoes_resumo"):
    print(t, spark.table(f"lake.gold.{t}").count(), "linhas")
c360.groupBy("segmento").count().orderBy("count", ascending=False).show()

### Servir: Gold (Iceberg) → Postgres
O Metabase não lê Iceberg. Ele lê o Postgres. `truncate=true` reaproveita a tabela, então os painéis não quebram a cada execução.

In [ ]:
for nome, df in (("vendas_diarias", vd), ("clientes_360", c360), ("avaliacoes_resumo", ares)):
    print("publicado:", L.publish_to_postgres(df, nome, schema="gold"))
L.pg_query("SELECT categoria, round(sum(receita)) AS receita FROM gold.vendas_diarias GROUP BY 1 ORDER BY 2 DESC LIMIT 5")

### Metabase (BI básico)
1. Abra <http://localhost:3000> e crie o usuário de administração.
2. *Adicionar banco* → **PostgreSQL** → host `postgres` (é o nome do serviço dentro do Docker), porta `5432`, banco `serving`, usuário `aula`, senha `aula123`.
3. *Sincronizar esquema* e explore o schema **gold**. Monte 3 perguntas:
   - **Receita por mês** (gráfico de linha): `vendas_diarias`, somar `receita` por mês de `data`.
   - **Clientes por segmento** (barras): `clientes_360`, contar por `segmento`.
   - **Nota média por categoria** (barras): `avaliacoes_resumo`.
4. Junte as três num *dashboard*.

## Parte B · Gold para o banco vetorial

Um modelo de linguagem não lê tabela de vendas. Ele precisa de **textos** que possa buscar por significado.
Aqui a Gold vira uma tabela de **chunks**: um pedaço de texto por linha, pronto para virar vetor.

O que a Gold faz *antes* de qualquer embedding (é aqui que se ganha ou se perde qualidade):
1. **Limpar** espaços e descartar texto muito curto.
2. **Mascarar PII** (CPF, telefone, e-mail): dado pessoal não pode ir parar num índice vetorial.
3. **ID estável** (`chunk_id`): o mesmo comentário sempre tem o mesmo id.
4. **`content_hash`**: impressão digital do texto. Se o hash não mudou, **não se paga embedding de novo**.
5. **Metadados** (categoria, nota, data) para *filtrar* a busca.

In [ ]:
# A máscara de PII, testada em texto de exemplo (o dado da aula não tem PII; o do mundo real tem)
from common import vetorial as V
for t in ["Meu CPF 123.456.789-09, fone (11) 98765-4321, joao@ex.com",
          "Pedido 100001 chegou em 3 dias por R$ 1234,56"]:
    print(V.mask_pii_py(t))

In [ ]:
chunks = V.build_chunks(spark)           # grava lake.gold.avaliacao_chunks (Iceberg)
print(chunks.count(), "chunks | textos únicos:", chunks.select("content_hash").distinct().count())
chunks.show(5, truncate=50)

Veja a diferença entre **chunks** e **textos únicos**: muitos clientes escrevem a mesma frase. Vamos calcular embedding **uma vez por texto único**, e não uma vez por linha.

### Gold → pgvector (sem embedding ainda)
`sync_to_pgvector` faz um *upsert*: texto novo entra; texto igual **mantém** o embedding; texto alterado **zera** o embedding para ser refeito; linha que sumiu da Gold é removida.

In [ ]:
V.sync_to_pgvector(chunks)
V.status()

Status esperado agora: todos os chunks em `pendentes` (nenhum embedding calculado). **A arquitetura está pronta**; o preenchimento dos vetores e a busca são o próximo módulo (há uma demonstração em `04_vetorial.ipynb`).

## Checkpoint
- [ ] Três tabelas em `gold` no Postgres e um dashboard no Metabase.
- [ ] `gold.avaliacao_chunks` no Iceberg e `vetorial.avaliacao_chunks` no Postgres, com `embedding` vazio.
- [ ] Você explica por que o Iceberg é a fonte da verdade e o pgvector é derivado.

➡️ Opcional: `04_vetorial.ipynb`